# 🚀 GPU-Accelerated YOLO11 Fine-Tuning — Coimbatore Urban Digital Twin
### FarmwiseAI Product Challenge 2026 — Team 24 (CCMC Ward 29)

This Google Colab notebook fine-tunes **YOLO11n** on an **NVIDIA T4 GPU (16 GB)** for 50–100 epochs with cosine learning rate scheduling and mosaic augmentations.

**Target Classes (12 categories)**:
`building`, `electric_pole`, `streetlight`, `commercial_sign`, `transformer`, `traffic_signal`, `road_sign`, `drain`, `sidewalk`, `waste_bin`, `bus_stop`, `other_urban_asset`.

## 1. Verify GPU Hardware Acceleration
Ensure your Colab runtime is set to **T4 GPU** (`Runtime` -> `Change runtime type` -> `T4 GPU`).

In [ ]:
!nvidia-smi

## 2. Install High-Performance Dependencies

In [ ]:
!pip install -q ultralytics onnx onnxruntime-gpu

## 3. Upload & Extract Dataset
Upload `urban_intelligence_dataset.zip` from your computer using the file explorer on the left or the file upload widget below:

In [ ]:
import os
import zipfile
from google.colab import files

# If file not uploaded yet, prompt upload:
if not os.path.exists('/content/urban_intelligence_dataset.zip'):
    print('Please upload urban_intelligence_dataset.zip:')
    uploaded = files.upload()

# Extract dataset
with zipfile.ZipFile('/content/urban_intelligence_dataset.zip', 'r') as zip_ref:
    zip_ref.extractall('/content/')

print('✅ Dataset successfully extracted to /content/dataset_splits!')
!ls -la /content/dataset_splits

## 4. Commencing GPU Fine-Tuning (50 Epochs)
Trained with:
- **Batch Size 16** on NVIDIA T4 GPU
- **Cosine Learning Rate Scheduler** (`cos_lr=True`)
- **Mosaic Augmentation** (`mosaic=1.0`) with `close_mosaic=10`
- **Loss Gains Tuned for Thin Urban Poles & Signs** (`box=7.5`, `cls=0.7`)

In [ ]:
from ultralytics import YOLO

# Load pretrained YOLO11n
model = YOLO('yolo11n.pt')

# Train on GPU
results = model.train(
    data='/content/dataset_splits/data.yaml',
    epochs=50,
    batch=16,
    imgsz=640,
    device=0,            # NVIDIA T4 GPU
    workers=4,
    optimizer='AdamW',
    lr0=0.001,
    lrf=0.01,
    cos_lr=True,
    box=7.5,
    cls=0.7,
    dfl=1.5,
    mosaic=1.0,
    mixup=0.15,
    close_mosaic=10,
    project='/content/urban_runs',
    name='urban_intel_colab',
    exist_ok=True,
    save=True,
    val=True
)

print('🎉 Training complete!')

## 5. Evaluate Validation Metrics (mAP, Precision, Recall)

In [ ]:
# Run comprehensive validation
metrics = model.val()

print('=' * 50)
print(f'mAP@50     : {metrics.box.map50:.4f}')
print(f'mAP@50-95  : {metrics.box.map:.4f}')
print(f'Precision  : {metrics.box.mp:.4f}')
print(f'Recall     : {metrics.box.mr:.4f}')
print('=' * 50)

# Display training curves and confusion matrix
from IPython.display import Image, display
run_dir = '/content/urban_runs/urban_intel_colab'
for img_name in ['results.png', 'confusion_matrix.png', 'BoxPR_curve.png']:
    p = os.path.join(run_dir, img_name)
    if os.path.exists(p):
        print(f'Displaying: {img_name}')
        display(Image(filename=p))

## 6. Export to 10 MB ONNX Runtime Artifact (Opset 18)

In [ ]:
# Export to ONNX
onnx_path = model.export(format='onnx', opset=18, dynamic=False, simplify=True)
print(f'✅ ONNX Exported to: {onnx_path}')
!ls -lh /content/urban_runs/urban_intel_colab/weights/

## 7. Package and Download Fine-Tuned Weights
This creates `fine_tuned_weights.zip` and triggers an automatic browser download.

In [ ]:
import shutil
from google.colab import files

# Package the weights and key plots
pkg_dir = '/content/colab_weights_package'
os.makedirs(pkg_dir, exist_ok=True)

weights_dir = '/content/urban_runs/urban_intel_colab/weights'
shutil.copy(os.path.join(weights_dir, 'best.pt'), os.path.join(pkg_dir, 'best.pt'))
shutil.copy(os.path.join(weights_dir, 'best.onnx'), os.path.join(pkg_dir, 'best.onnx'))
shutil.copy('/content/urban_runs/urban_intel_colab/results.csv', os.path.join(pkg_dir, 'results.csv'))
shutil.copy('/content/urban_runs/urban_intel_colab/results.png', os.path.join(pkg_dir, 'results.png'))

shutil.make_archive('/content/fine_tuned_weights', 'zip', pkg_dir)

print('Downloading fine_tuned_weights.zip...')
files.download('/content/fine_tuned_weights.zip')